<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-11-cost-and-performance/lesson-11.1-ai-app-costs/notebooks/exercises-11_1.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 11.1: Understanding AI Application Costs

*Module 11 · LLM Optimization & Self-Hosting*

> LLM APIs price by tokens, not requests. A 100-word query and a 10,000-word document cost 100x differently. This lesson builds the pricing models, cost calculators, and BigQuery dashboards that give you complete visibility into what your AI app actually costs — across every provider, every model, every team.

---

## About this notebook

This notebook contains the **5 runnable code examples** from the published lesson `lesson-11.1.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Multi-Provider Pricing Database — What Everything Actually Costs — `pricing_db.py`
2. Step 2 — Cost Calculator — What Will This Actually Cost at Scale? — `cost_calculator.py`
3. Step 3 — Request-Level Cost Logger — Track Every Call — `cost_logger.py`
4. Step 4 — BigQuery Cost Dashboard — SQL Queries That Save Money — `bigquery_dashboard.sql`
5. Step 5 — Cost Optimization Analyzer — Find and Fix the Biggest Savings — `cost_optimizer.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q anthropic>=0.40.0 openai requests datasets


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Multi-Provider Pricing Database — What Everything Actually Costs

Real pricing for every model you might use. Updated for 2025.

**`pricing_db.py`** · _Block 1 of 5_

MULTI-PROVIDER PRICING DATABASE — All prices per 1M tokens (USD).


In [ ]:
# =============================================
# MULTI-PROVIDER PRICING DATABASE
# All prices per 1M tokens (USD).
# Updated: 2025. Check provider docs for latest.
# =============================================

from dataclasses import dataclass

@dataclass
class ModelPricing:
    provider: str
    model: str
    input_per_1m: float      # USD per 1M input tokens
    output_per_1m: float     # USD per 1M output tokens
    context_window: int      # max tokens
    category: str            # "fast", "balanced", "frontier"

PRICING_DB: list[ModelPricing] = [
    # ── Google Gemini ──
    ModelPricing("google", "gemini-2.0-flash",      0.10,  0.40,   1_000_000, "fast"),
    ModelPricing("google", "gemini-2.5-flash",      0.15,  0.60,   1_000_000, "fast"),
    ModelPricing("google", "gemini-2.5-pro",        1.25,  10.00,  1_000_000, "frontier"),
    ModelPricing("google", "text-embedding-004",    0.006, 0.0,    2048,      "embedding"),
    
    # ── OpenAI ──
    ModelPricing("openai", "gpt-4o-mini",           0.15,  0.60,   128_000,   "fast"),
    ModelPricing("openai", "gpt-4o",                2.50,  10.00,  128_000,   "frontier"),
    ModelPricing("openai", "gpt-4.1",               2.00,  8.00,   1_000_000, "frontier"),
    ModelPricing("openai", "gpt-4.1-mini",          0.40,  1.60,   1_000_000, "balanced"),
    ModelPricing("openai", "text-embedding-3-small", 0.02,  0.0,    8191,      "embedding"),
    
    # ── Anthropic ──
    ModelPricing("anthropic", "claude-sonnet-4",     3.00,  15.00,  200_000,   "frontier"),
    ModelPricing("anthropic", "claude-haiku-3.5",    0.80,  4.00,   200_000,   "balanced"),
]

def get_pricing(model: str) -> ModelPricing:
    for p in PRICING_DB:
        if p.model == model:
            return p
    return None

# ── Print comparison ──
print("LLM Pricing (per 1M tokens, USD):\n")
print(f"  {'Model':28s} {'Input':>7s} {'Output':>8s} {'Out/In':>7s}  {'Category'}")
print(f"  {'─'*70}")
for p in PRICING_DB:
    ratio = p.output_per_1m / p.input_per_1m if p.input_per_1m > 0 else 0
    print(f"  {p.model:28s} ${p.input_per_1m:>6.3f} ${p.output_per_1m:>7.2f}   {ratio:>5.1f}x  {p.category}")


> **What just happened?** A complete pricing database across 3 providers (Google, OpenAI, Anthropic) and 4 categories (fast, balanced, frontier, embedding). Key insight: output tokens cost 2-5x more than input tokens. Gemini Flash output ($0.40/M) is 4x its input ($0.10/M). Claude Sonnet output ($15/M) is 5x its input ($3/M). This means a chatbot that generates long responses costs much more than one that generates concise answers — even with the same input.


### Step 2 · Cost Calculator — What Will This Actually Cost at Scale?

Estimate per-request cost. Project monthly spend. Compare models side by side.

**`cost_calculator.py`** · _Block 2 of 5_

COST CALCULATOR — Estimate per-request and at-scale costs.


In [ ]:
# =============================================
# COST CALCULATOR
# Estimate per-request and at-scale costs.
# Compare models. Find the cheapest option.
# =============================================

class CostCalculator:
    """Calculate and compare LLM costs at any scale."""
    
    def __init__(self, pricing_db: list[ModelPricing] = None):
        self.db = {p.model: p for p in (pricing_db or PRICING_DB)}
    
    def per_request(self, model: str, input_tokens: int, output_tokens: int) -> dict:
        """Cost of a single request."""
        p = self.db.get(model)
        if not p: return {"error": f"Unknown model: {model}"}
        
        input_cost = input_tokens * p.input_per_1m / 1_000_000
        output_cost = output_tokens * p.output_per_1m / 1_000_000
        total = input_cost + output_cost
        
        return {
            "model": model, "input_tokens": input_tokens, "output_tokens": output_tokens,
            "input_cost_usd": round(input_cost, 6),
            "output_cost_usd": round(output_cost, 6),
            "total_usd": round(total, 6),
            "total_inr": round(total * 84, 4),
        }
    
    def monthly(self, model: str, daily_requests: int,
                avg_input: int = 500, avg_output: int = 300) -> dict:
        """Project monthly cost."""
        per_req = self.per_request(model, avg_input, avg_output)
        if "error" in per_req: return per_req
        
        monthly_usd = per_req["total_usd"] * daily_requests * 30
        return {
            "model": model,
            "daily_requests": daily_requests,
            "cost_per_request": per_req["total_usd"],
            "daily_usd": round(monthly_usd / 30, 2),
            "monthly_usd": round(monthly_usd, 2),
            "monthly_inr": round(monthly_usd * 84, 0),
        }
    
    def compare(self, models: list[str], input_tokens: int, output_tokens: int,
                daily_requests: int = 1000) -> list[dict]:
        """Compare costs across models for the same workload."""
        results = []
        for model in models:
            m = self.monthly(model, daily_requests, input_tokens, output_tokens)
            if "error" not in m:
                results.append(m)
        
        # Sort by cost
        results.sort(key=lambda r: r["monthly_usd"])
        
        # Calculate savings vs most expensive
        if results:
            most_expensive = results[-1]["monthly_usd"]
            for r in results:
                r["savings_vs_max"] = round((1 - r["monthly_usd"] / most_expensive) * 100) if most_expensive > 0 else 0
        
        return results
    
    def rag_cost(self, model: str, query_tokens: int = 50,
                 context_tokens: int = 3000, output_tokens: int = 400,
                 embed_model: str = "text-embedding-004",
                 chunks_retrieved: int = 5) -> dict:
        """Full cost of a RAG request (embed + retrieve + generate)."""
        # Embedding cost (query embedding)
        embed_p = self.db.get(embed_model)
        embed_cost = query_tokens * embed_p.input_per_1m / 1_000_000 if embed_p else 0
        
        # LLM cost (query + retrieved context → answer)
        total_input = query_tokens + context_tokens
        llm = self.per_request(model, total_input, output_tokens)
        
        total = embed_cost + llm.get("total_usd", 0)
        
        return {
            "embed_cost": round(embed_cost, 6),
            "llm_cost": llm.get("total_usd", 0),
            "total_usd": round(total, 6),
            "context_pct": round(context_tokens / total_input * 100),
            "note": f"{context_tokens} context tokens = {llm.get('input_cost_usd',0) - query_tokens * self.db[model].input_per_1m / 1_000_000:.6f} USD of 'hidden' input cost",
        }

# ═══════════════════════════════════════════
# DEMOS
# ═══════════════════════════════════════════

calc = CostCalculator()

# 1. Single request cost
print("1. Single request cost:\n")
for model in ["gemini-2.0-flash", "gpt-4o-mini", "gpt-4o", "claude-sonnet-4"]:
    r = calc.per_request(model, 500, 300)
    print(f"  {model:25s} ${r['total_usd']:.6f}  (₹{r['total_inr']:.4f})")

# 2. Monthly projection at scale
print(f"\n2. Monthly cost at 5,000 requests/day:\n")
comparison = calc.compare(
    ["gemini-2.0-flash", "gpt-4o-mini", "gpt-4.1-mini", "gpt-4o", "claude-sonnet-4"],
    input_tokens=500, output_tokens=300, daily_requests=5000,
)
for r in comparison:
    print(f"  {r['model']:25s} ${r['monthly_usd']:>8.2f}/mo  ₹{r['monthly_inr']:>8,.0f}  ({r['savings_vs_max']}% cheaper)")

# 3. RAG hidden cost
print(f"\n3. RAG hidden cost (3,000 context tokens per query):\n")
rag = calc.rag_cost("gemini-2.0-flash", query_tokens=50, context_tokens=3000, output_tokens=400)
print(f"  Embed: ${rag['embed_cost']:.6f}  LLM: ${rag['llm_cost']:.6f}  Total: ${rag['total_usd']:.6f}")
print(f"  Context is {rag['context_pct']}% of input tokens — the 'hidden' cost of RAG")


> **What just happened?** CostCalculator with 4 functions: per_request (exact cost for one call), monthly (project daily × 30), compare (rank models by cost for the same workload, show % savings), rag_cost (full RAG cost: embedding + context tokens + generation). Key finding: at 5,000 req/day, Gemini Flash costs ~₹630/month while Claude Sonnet costs ~₹50,400/month — 80x difference for the same task. RAG's "hidden cost": 3,000 context tokens are 98% of input but most people only think about the 50-token query.


### Step 3 · Request-Level Cost Logger — Track Every Call

Log tokens + cost for every LLM call. Export to BigQuery for analysis.

**`cost_logger.py`** · _Block 3 of 5_

REQUEST-LEVEL COST LOGGER — Log every LLM call with tokens and cost.


In [ ]:
# =============================================
# REQUEST-LEVEL COST LOGGER
# Log every LLM call with tokens and cost.
# Structured JSON → Cloud Logging → BigQuery.
# =============================================

import json, time, uuid
from datetime import datetime

class CostLogger:
    """Log every LLM call with cost metadata."""
    
    def __init__(self, calculator: CostCalculator):
        self.calc = calculator
        self.logs: list[dict] = []
    
    def log(self, *, model: str, input_tokens: int, output_tokens: int,
            latency_ms: float, team: str = "default",
            endpoint: str = "/v1/chat", cached: bool = False,
            request_id: str = "") -> dict:
        """Log a single LLM call."""
        cost = self.calc.per_request(model, input_tokens, output_tokens)
        
        entry = {
            "timestamp": datetime.utcnow().isoformat() + "Z",
            "request_id": request_id or str(uuid.uuid4())[:8],
            "team": team,
            "model": model,
            "provider": self.calc.db[model].provider if model in self.calc.db else "unknown",
            "endpoint": endpoint,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": input_tokens + output_tokens,
            "cost_usd": cost["total_usd"],
            "latency_ms": latency_ms,
            "cached": cached,
            "cost_per_token": round(cost["total_usd"] / max(input_tokens + output_tokens, 1) * 1000, 6),
        }
        
        self.logs.append(entry)
        
        # Print as structured JSON (Cloud Logging auto-parses this)
        print(json.dumps({"severity": "INFO", "message": "llm_call", **entry}))
        
        return entry
    
    def summary(self) -> dict:
        """Summarize all logged calls."""
        if not self.logs: return {}
        total_cost = sum(l["cost_usd"] for l in self.logs)
        total_tokens = sum(l["total_tokens"] for l in self.logs)
        cached = sum(1 for l in self.logs if l["cached"])
        
        # Cost by team
        by_team = {}
        for l in self.logs:
            by_team.setdefault(l["team"], 0)
            by_team[l["team"]] += l["cost_usd"]
        
        # Cost by model
        by_model = {}
        for l in self.logs:
            by_model.setdefault(l["model"], {"calls": 0, "cost": 0})
            by_model[l["model"]]["calls"] += 1
            by_model[l["model"]]["cost"] += l["cost_usd"]
        
        return {"total_calls": len(self.logs), "total_tokens": total_tokens,
                "total_cost_usd": round(total_cost, 4), "cached_pct": round(cached/len(self.logs)*100),
                "by_team": by_team, "by_model": by_model}

# ── Demo ──
logger = CostLogger(calc)

logger.log(model="gemini-2.0-flash", input_tokens=500, output_tokens=300, latency_ms=800, team="product")
logger.log(model="gemini-2.5-pro", input_tokens=5000, output_tokens=2000, latency_ms=12000, team="research")
logger.log(model="gemini-2.0-flash", input_tokens=200, output_tokens=100, latency_ms=3, team="product", cached=True)

print(f"\nSummary: {json.dumps(logger.summary(), indent=2)}")


> **What just happened?** CostLogger records every LLM call with: timestamp, request_id, team, model, provider, tokens (input/output/total), cost in USD, latency, and cache status. Output is structured JSON that Cloud Logging auto-parses. summary() aggregates: total cost, cost by team, cost by model, cache hit rate. This is the data source for all downstream analysis — BigQuery dashboards, alerts, and optimization.


### Step 4 · BigQuery Cost Dashboard — SQL Queries That Save Money

Route Cloud Logging → BigQuery. Run SQL to find your biggest cost drivers.

**`bigquery_dashboard.sql`** · _Block 4 of 5_


In [ ]:
-- ═══════════════════════════════════════════
-- BIGQUERY COST DASHBOARD QUERIES
-- Route: Cloud Logging → Log Sink → BigQuery
-- Table: `project.dataset.llm_costs`
-- ═══════════════════════════════════════════

-- Setup: Create a Log Sink to BigQuery
-- gcloud logging sinks create llm-costs-sink \
--   bigquery.googleapis.com/projects/PROJECT/datasets/ai_costs \
--   --log-filter='jsonPayload.message="llm_call"'

-- ── Query 1: Daily spend by team ──
SELECT
  DATE(timestamp) AS day,
  jsonPayload.team AS team,
  COUNT(*) AS calls,
  SUM(CAST(jsonPayload.cost_usd AS FLOAT64)) AS total_usd,
  ROUND(SUM(CAST(jsonPayload.cost_usd AS FLOAT64)) * 84, 2) AS total_inr,
  SUM(CAST(jsonPayload.total_tokens AS INT64)) AS total_tokens
FROM `project.dataset.llm_costs`
WHERE timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
GROUP BY day, team
ORDER BY day DESC, total_usd DESC;

-- ── Query 2: Cost by model (which model burns the most?) ──
SELECT
  jsonPayload.model AS model,
  jsonPayload.provider AS provider,
  COUNT(*) AS calls,
  ROUND(SUM(CAST(jsonPayload.cost_usd AS FLOAT64)), 4) AS total_usd,
  ROUND(AVG(CAST(jsonPayload.cost_usd AS FLOAT64)), 6) AS avg_per_call,
  ROUND(AVG(CAST(jsonPayload.latency_ms AS FLOAT64)), 0) AS avg_latency_ms,
  ROUND(AVG(CAST(jsonPayload.total_tokens AS INT64)), 0) AS avg_tokens
FROM `project.dataset.llm_costs`
WHERE timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
GROUP BY model, provider
ORDER BY total_usd DESC;

-- ── Query 3: Cache savings (how much did caching save?) ──
SELECT
  COUNTIF(jsonPayload.cached = 'true') AS cached_calls,
  COUNTIF(jsonPayload.cached = 'false') AS llm_calls,
  ROUND(COUNTIF(jsonPayload.cached = 'true') / COUNT(*) * 100, 1) AS cache_hit_pct,
  ROUND(SUM(IF(jsonPayload.cached = 'false',
    CAST(jsonPayload.cost_usd AS FLOAT64), 0)), 4) AS actual_spend,
  ROUND(SUM(CAST(jsonPayload.cost_usd AS FLOAT64)), 4) AS would_have_spent,
  ROUND(SUM(IF(jsonPayload.cached = 'true',
    CAST(jsonPayload.cost_usd AS FLOAT64), 0)), 4) AS saved_by_cache
FROM `project.dataset.llm_costs`
WHERE timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY);

-- ── Query 4: Expensive outliers (find the costly requests) ──
SELECT
  jsonPayload.request_id,
  jsonPayload.team,
  jsonPayload.model,
  CAST(jsonPayload.input_tokens AS INT64) AS input_tokens,
  CAST(jsonPayload.output_tokens AS INT64) AS output_tokens,
  CAST(jsonPayload.cost_usd AS FLOAT64) AS cost_usd,
  timestamp
FROM `project.dataset.llm_costs`
WHERE CAST(jsonPayload.cost_usd AS FLOAT64) > 0.05  -- requests costing > 5 cents
ORDER BY cost_usd DESC
LIMIT 20;

-- ── Query 5: Hourly cost trend (spot anomalies) ──
SELECT
  TIMESTAMP_TRUNC(timestamp, HOUR) AS hour,
  COUNT(*) AS calls,
  ROUND(SUM(CAST(jsonPayload.cost_usd AS FLOAT64)), 4) AS cost_usd
FROM `project.dataset.llm_costs`
WHERE timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 48 HOUR)
GROUP BY hour
ORDER BY hour;


> **What just happened?** Five BigQuery queries that answer the critical cost questions: (1) Daily spend by team — who's spending how much? (2) Cost by model — which model burns the most? Average cost per call. (3) Cache savings — how much money did caching save? (4) Expensive outliers — find requests costing >5¢ (probably huge context windows or Pro model misuse). (5) Hourly trend — spot cost spikes in real time. Setup: one gcloud command creates a Log Sink from Cloud Logging to BigQuery.


### Step 5 · Cost Optimization Analyzer — Find and Fix the Biggest Savings

Analyze your spending and recommend specific optimizations with estimated savings.

**`cost_optimizer.py`** · _Block 5 of 5_

COST OPTIMIZATION ANALYZER — Analyze spending patterns and recommend


In [ ]:
# =============================================
# COST OPTIMIZATION ANALYZER
# Analyze spending patterns and recommend
# specific optimizations with $ savings.
# =============================================

class CostOptimizer:
    """Analyze cost logs and recommend optimizations."""
    
    def __init__(self, calculator: CostCalculator):
        self.calc = calculator
    
    def analyze(self, logs: list[dict]) -> list[dict]:
        """Analyze cost logs and return optimization recommendations."""
        recommendations = []
        
        # ── Check 1: Model downgrade opportunities ──
        model_costs = {}
        for l in logs:
            model_costs.setdefault(l["model"], {"calls": 0, "cost": 0, "avg_tokens": []})
            model_costs[l["model"]]["calls"] += 1
            model_costs[l["model"]]["cost"] += l["cost_usd"]
            model_costs[l["model"]]["avg_tokens"].append(l["total_tokens"])
        
        for model, data in model_costs.items():
            p = self.calc.db.get(model)
            if p and p.category == "frontier":
                avg_tok = sum(data["avg_tokens"]) / len(data["avg_tokens"])
                if avg_tok < 1000:  # short queries on expensive model
                    savings = data["cost"] * 0.85  # ~85% cheaper on Flash
                    recommendations.append({
                        "type": "model_downgrade",
                        "priority": "HIGH",
                        "description": f"{data['calls']} calls to {model} with avg {avg_tok:.0f} tokens — short queries don't need a frontier model",
                        "action": f"Route queries under 1K tokens to gemini-2.0-flash",
                        "estimated_savings_usd": round(savings, 4),
                    })
        
        # ── Check 2: Cache hit rate too low ──
        cached = sum(1 for l in logs if l.get("cached"))
        hit_rate = cached / len(logs) * 100 if logs else 0
        if hit_rate < 20:
            potential = sum(l["cost_usd"] for l in logs) * 0.3  # 30% cache hit rate target
            recommendations.append({
                "type": "improve_caching",
                "priority": "HIGH",
                "description": f"Cache hit rate is {hit_rate:.0f}% — target 30-40%",
                "action": "Enable semantic cache (10.4), warm with top queries",
                "estimated_savings_usd": round(potential, 4),
            })
        
        # ── Check 3: Large context windows ──
        large_ctx = [l for l in logs if l["input_tokens"] > 5000]
        if len(large_ctx) > len(logs) * 0.1:
            excess_cost = sum(l["cost_usd"] for l in large_ctx) * 0.4
            recommendations.append({
                "type": "reduce_context",
                "priority": "MEDIUM",
                "description": f"{len(large_ctx)} calls ({len(large_ctx)/len(logs)*100:.0f}%) have >5K input tokens",
                "action": "Reduce RAG chunk count, summarize context before sending to LLM",
                "estimated_savings_usd": round(excess_cost, 4),
            })
        
        recommendations.sort(key=lambda r: r["estimated_savings_usd"], reverse=True)
        return recommendations

# ── Demo ──
optimizer = CostOptimizer(calc)
recs = optimizer.analyze(logger.logs)

print("Cost Optimization Recommendations:\n")
for i, r in enumerate(recs, 1):
    print(f"  {i}. [{r['priority']}] {r['type']}")
    print(f"     {r['description']}")
    print(f"     Action: {r['action']}")
    print(f"     Estimated savings: ${r['estimated_savings_usd']:.4f}\n")


> **What just happened?** CostOptimizer analyzes your cost logs and generates prioritized recommendations: (1) Model downgrade — short queries (<1K tokens) on frontier models → switch to Flash, save ~85%. (2) Improve caching — hit rate below 20% → enable semantic cache, save ~30%. (3) Reduce context — too many large context windows → fewer RAG chunks or pre-summarize, save ~40% of those calls. Each recommendation includes: priority, description, specific action, and estimated dollar savings. Run this monthly. The top recommendation alone typically saves 30-50% of your bill.


---

## Wrap-up

You've walked through all 5 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
